# Mixture of Experts (MoE) Implementation Questions

This notebook contains questions about implementing Mixture of Experts layers in PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Question 1: Basic MoE Implementation

**Background:** A Mixture of Experts (MoE) layer routes each input token to one or more expert networks. The router network determines which experts should process each token.

**Task:** Implement a basic MoE block with the following specifications:
- Multiple expert networks (each is an MLP)
- A router that selects which expert processes each token
- Top-k routing (select k experts per token)

HINT: I suggest just doing top-1 first, then think about top-k.

Complete the implementation below:

In [ ]:
class MLP(nn.Module):
    
    def __init__(self, hidden_size, ffn_hidden_size):
        super().__init__()
        # TODO: Implement a 2-layer feedforward network
        pass
    
    def forward(self, x):
        # x: (batch_size, seq_len, hidden_size)
        # TODO: Implement forward pass
        pass


class MoE(nn.Module):
    """Mixture of Experts layer."""
    
    def __init__(self, num_experts, hidden_size, ffn_hidden_size, top_k=1):
        super().__init__()
        # TODO: Create router network (projects hidden_size -> num_experts)
        # TODO: Create list of expert networks
        pass
    
    def forward(self, x):
        # x: (batch_size, seq_len, hidden_size)
        
        # SUGGESTED WORKFLOW. FEEL FREE TO IGNORE.
        # TODO: Step 1 - Get routing scores from router network
        # TODO: Step 2 - Apply softmax to get routing probabilities
        # TODO: Step 3 - Select top-k experts and their weights
        # TODO: Step 4 - For each selected expert, process the input
        # TODO: Step 5 - Combine expert outputs using routing weights
        
        pass

Now let's test the above implementation. 

Extra credit: Test logits and grads for NaN/Inf

In [ ]:
x = torch.randn(2, 3, 5)  # (batch=2, seq_len=3, hidden=5)
moe = MoE(num_experts=4, hidden_size=5, ffn_hidden_size=20)
output = moe(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

## Question 2: Load Balancing

**Problem:** Without constraints, the router may send all tokens to a small subset of experts, leaving other experts underutilized.

**Task:** Describe 2-3 strategies to encourage balanced expert usage. Consider:
- How would you detect imbalanced usage?
- What mechanisms could encourage balance?
- What are the tradeoffs of different approaches?

## Question 2a: Implement Auxiliary Load Balancing Loss

**Reference:** [GShard paper (Section 2.2)](https://arxiv.org/pdf/2006.16668)

**Task:** Implement the auxiliary loss function that encourages balanced expert usage:

The loss penalizes uneven distribution of tokens across experts:
```
L_balance = α · num_experts · Σ(f_i · P_i)
```
where:
- `f_i` = fraction of tokens assigned to expert i
- `P_i` = mean routing probability for expert i
- `α` = loss coefficient (e.g., 0.01)

Complete the implementation:

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_size, ffn_hidden_size):
        super().__init__()
        # Same as before
        pass
    
    def forward(self, x):
        # Same as before
        pass


class MoE(nn.Module):
    def __init__(self, num_experts, hidden_size, ffn_hidden_size, top_k=2, aux_loss_coef=0.01):
        super().__init__()
        self.num_experts = num_experts
        self.aux_loss_coef = aux_loss_coef
        
        # TODO: Same initialization as Question 1
        pass
    
    def forward(self, x):
        # x: (batch_size, seq_len, hidden_size)
        
        # TODO: Same routing and expert processing as Question 1
        
        # TODO: Calculate auxiliary load balancing loss
        # Hints: You need to compute:
        # 1. Fraction of tokens routed to each expert (f_i)
        # 2. Mean routing probability for each expert (P_i)
        # 3. Combine into loss: aux_loss_coef * num_experts * sum(f_i * P_i)
        
        if self.training:
            # Return both output and auxiliary loss during training
            return output, aux_loss
        
        return output

In [ ]:
# Test code
x = torch.randn(2, 3, 5)
moe = MoE(num_experts=4, hidden_size=5, ffn_hidden_size=20)
moe.train()

output, aux_loss = moe(x)
print(f"Output shape: {output.shape}")
print(f"Auxiliary loss: {aux_loss.item():.4f}")

## Question 3: Expert Parallelism with Data Parallelism

**Scenario:** You have a large MoE model that doesn't fit on a single GPU. You want to use:
- **Data parallelism**: Split the batch across GPUs
- **Expert parallelism**: Split experts across GPUs

**Example setup (2 GPUs, 4 experts, batch size B):**
- GPU 0: samples [0 to B/2-1], experts [0, 1]
- GPU 1: samples [B/2 to B-1], experts [2, 3]

**Problem:** Samples on GPU 0 may need experts on GPU 1, requiring cross-GPU communication.

### Part A: Communication Operations

Given these PyTorch distributed operations:

```python
# all_gather: Gather tensors from all GPUs
# Input:  GPU 0: [1, 2], GPU 1: [3, 4]
# Output: Both GPUs: [1, 2, 3, 4]

# all_reduce: Sum tensors across GPUs
# Input:  GPU 0: [1, 2], GPU 1: [3, 4]
# Output: Both GPUs: [4, 6]

# scatter: Split tensor across GPUs
# Input:  Both GPUs: [1, 2, 3, 4]
# Output: GPU 0: [1, 2], GPU 1: [3, 4]
```

**Task:** Fill in the missing communication operations and tensor shapes in the code below:

In [ ]:
class MoE(nn.Module):
    def __init__(self, num_experts, hidden_size, ffn_hidden_size):
        super().__init__()
        
        # Router is shared across all GPUs
        self.router = nn.Linear(hidden_size, num_experts)
        
        # Each GPU stores a subset of experts
        self.local_experts_ids = # TODO: list of expert IDs on this GPU
        self.local_experts = nn.ModuleList([
            MLP(hidden_size, ffn_hidden_size) 
            for _ in self.local_experts_ids
        ])
    
    def forward(self, x):
        # x: (local_batch_size, seq_len, hidden_size) - only local samples
        
        # Get routing decisions using local data
        scores = self.router(x)
        probs = F.softmax(scores, dim=-1)
        routing_weights, expert_ids = probs.max(dim=-1)
        
        # Step 1: Gather expert assignments from all GPUs
        # TODO: What operation? What shape?
        global_expert_ids = torch.empty(
            (???),  # TODO: Fill in shape
            dtype=expert_ids.dtype,
            device=expert_ids.device
        )
        torch.distributed.???(
            global_expert_ids,  # output
            expert_ids          # input from this GPU
        )
        
        # Step 2: Gather input tensors from all GPUs
        # TODO: What operation? What shape?
        global_x = torch.empty(
            (???),  # TODO: Fill in shape
            dtype=x.dtype,
            device=x.device
        )
        torch.distributed.???(
            global_x,  # output
            x          # input from this GPU
        )
        
        # Step 3: Each GPU processes tokens assigned to its local experts
        output_total = torch.zeros_like(global_x)
        
        for i, expert in enumerate(self.local_experts):
            local_expert_id = self.local_experts_ids[i]
            
            # Find tokens routed to this expert
            mask = (global_expert_ids == local_expert_id)
            expert_input = global_x[mask]
            
            # Process with expert
            expert_output = expert(expert_input)
            
            # Store in output
            output_total[mask] = expert_output
        
        # Step 4: Return only the local portion of outputs
        # TODO: What operation? What shape?
        output_local = torch.empty(
            (???),  # TODO: Fill in shape
            dtype=output_total.dtype,
            device=output_total.device
        )
        torch.distributed.???(
            output_local,   # output
            output_total    # input
        )
        
        # Apply routing weights
        output_local = output_local * routing_weights.unsqueeze(-1)
        
        return output_local

### Part B: Load Balancing and GPU Efficiency

**Question:** How does load balancing improve GPU utilization and efficiency in this expert-parallel setup?

Consider:
- What happens if one expert gets 90% of tokens?
- How does this affect GPU memory and compute?
- What's the impact on training time?